## 1. Reading in the data
We have to first read the data from the *data* directory. We will use -9999 for missing values and convert them to N/A. Additionally, we will use whitespace as the delimiter. 

In [1]:
# All imports needed in this notebook in one cell to simplify importing
import pandas as pd
from math import isnan

In [2]:
# Define path for the data file
file = r'data/2315676.txt'
# Read the date using whitespace as the delimiter, -9999 as missing values, 0th row as the header and
# additionally skip 1st row where there is just dashes
data = pd.read_csv(file, delim_whitespace=True, na_values=[-9999], header=[0], skiprows=[1])
# Print data
print(data)
# Get the number of original data rows
orig_n = len(data)

                 STATION      STATION_NAME      DATE  TAVG  TMAX  TMIN
0      GHCND:FI000007501  SODANKYLA-AWS-FI  19080101   NaN   2.0 -37.0
1      GHCND:FI000007501  SODANKYLA-AWS-FI  19080102   NaN   6.0 -26.0
2      GHCND:FI000007501  SODANKYLA-AWS-FI  19080103   NaN   7.0 -27.0
3      GHCND:FI000007501  SODANKYLA-AWS-FI  19080104   NaN  -3.0 -27.0
4      GHCND:FI000007501  SODANKYLA-AWS-FI  19080105   NaN   4.0 -36.0
...                  ...               ...       ...   ...   ...   ...
41060  GHCND:FI000007501  SODANKYLA-AWS-FI  20201003  47.0  51.0   NaN
41061  GHCND:FI000007501  SODANKYLA-AWS-FI  20201004  43.0  47.0  37.0
41062  GHCND:FI000007501  SODANKYLA-AWS-FI  20201005  42.0   NaN  37.0
41063  GHCND:FI000007501  SODANKYLA-AWS-FI  20201006  45.0  46.0  43.0
41064  GHCND:FI000007501  SODANKYLA-AWS-FI  20201007  47.0  50.0  44.0

[41065 rows x 6 columns]


## 2. Calculating estimated temperatures
We can immediately see that there are some missing values in the TAVG column. For those columns, we can estimate an average by averaging the TMAX and TMIN values. However, there are some rows where either of those values is missing. For rows where either of the TMAX and TMIN values is missing, the return value for the calculation will also be N/A. 

In [3]:
# Calculate estimated average for all rows. 
data['ESTAVG'] = (data['TMAX'] + data['TMIN'])/2
# Take copy of the data for a future sanity check
data_copy = data.copy()
# Drop rows where both the TAVG and ESTAVG values are missing
data = data.dropna(axis=0, how='all', subset=['TAVG', 'ESTAVG'])
# Get the number of rows after dropping the N/A rows
fil_n = len(data)
print(data)

                 STATION      STATION_NAME      DATE  TAVG  TMAX  TMIN  ESTAVG
0      GHCND:FI000007501  SODANKYLA-AWS-FI  19080101   NaN   2.0 -37.0   -17.5
1      GHCND:FI000007501  SODANKYLA-AWS-FI  19080102   NaN   6.0 -26.0   -10.0
2      GHCND:FI000007501  SODANKYLA-AWS-FI  19080103   NaN   7.0 -27.0   -10.0
3      GHCND:FI000007501  SODANKYLA-AWS-FI  19080104   NaN  -3.0 -27.0   -15.0
4      GHCND:FI000007501  SODANKYLA-AWS-FI  19080105   NaN   4.0 -36.0   -16.0
...                  ...               ...       ...   ...   ...   ...     ...
41060  GHCND:FI000007501  SODANKYLA-AWS-FI  20201003  47.0  51.0   NaN     NaN
41061  GHCND:FI000007501  SODANKYLA-AWS-FI  20201004  43.0  47.0  37.0    42.0
41062  GHCND:FI000007501  SODANKYLA-AWS-FI  20201005  42.0   NaN  37.0     NaN
41063  GHCND:FI000007501  SODANKYLA-AWS-FI  20201006  45.0  46.0  43.0    44.5
41064  GHCND:FI000007501  SODANKYLA-AWS-FI  20201007  47.0  50.0  44.0    47.0

[40882 rows x 7 columns]


We can see that the number of rows we filtered out is

In [4]:
print(orig_n - fil_n)

183


We can confirm that the number of dropped rows make sense by selecting those rows from the *data* in a different way. We will select the rows where both of those are missing. Then as we use *len()* for the selection we should have 183 rows. Let's do it! 

In [5]:
# Select rows where both TAVG and ESTAVG are missing
len(data_copy.loc[(data_copy['TAVG'].isna()) & (data_copy['ESTAVG'].isna())])

183

which confirms, that the *dropna* worked as it should.

## 3. Temperature unit conversion
In the original data, all the temperatures are given in Fahrenheit so we have to convert them to Celsius using the function we'll define here next.

In [6]:
def fahr_to_celsius(temp_fahrenheit):
    """
    Converts temperature values given in Fahrenheit (param: temp_fahrenheit) to Celsius degrees.
    Usage: fahr_to_celsius(48)
    """
    converted_temp = (temp_fahrenheit - 32) / 1.8
    return converted_temp

Then we can use the function for the average temperature columns. Let's define new columns for them with the prefix *C_*:

In [7]:
# Convert AVG to Celsius
data['C_TAVG'] = data['TAVG'].apply(fahr_to_celsius)
# Convert ESTAVG to Celsius
data['C_ESTAVG'] = data['ESTAVG'].apply(fahr_to_celsius)
#print(data[''])

Then we have to combine the two Series for average temperature favoring the actual average over the estimated. I wrote a function below for the task, but apparently there's *Pandas.Series.combine_first()*, which does absolutely the same thing

In [8]:
def combine_averages(avg, est):
    """
    Combine two Pandas.Series assuming the first is the actual average temperature and the second is 
    the estimated temperature. 
    """
    ls = []
    zipped = zip(avg,est)
    for (a,e) in zipped:
        if not isnan(a):
            ls.append(a)
        else:
            ls.append(e)
    return pd.Series(ls)
    


In [9]:
data['C_COMBINED_AVERAGES'] = combine_averages(data['C_TAVG'], data['C_ESTAVG'])

In [10]:
test = data['C_TAVG'].combine_first(data['C_ESTAVG'])

In [11]:
tuple(test) == tuple(combine_averages(data['C_TAVG'], data['C_ESTAVG']))

True

as can be seen. Let's print out the data:

In [12]:
data['C_COMBINED_AVERAGES'].isna().sum()

183

In [13]:
len(data.loc[(data['C_TAVG'].isna()) & (data['C_ESTAVG'].isna())])

0